# Tugas Praktikum Big Data Pertemuan 9 "Kopi Nusantara"  
Sholahuddin Ahmad 250506070

## Bagian A. Membangun Katalog Menu (Create)

In [1]:
from pymongo import MongoClient

# Hubungkan ke MongoDB
client = MongoClient("mongodb://localhost:27017/")
db_kn = client["kopi_nusantara"]
koleksi_menu = db_kn["menu"]

# Bersihkan collection agar eksperimen bersih & reproducible
koleksi_menu.delete_many({})

# 12 Dokumen Menu (6 Kopi & 6 Makanan)
menu_kopi_nusantara = [
    # Kategori Kopi (memiliki 'tingkat_kepahitan')
    {"nama_menu": "Espresso Nusantara", "kategori": "Kopi", "harga": 18000, "tingkat_kepahitan": 5, "stok": 50, "rekomendasi_barista": True},
    {"nama_menu": "Kopi Susu Gula Aren", "kategori": "Kopi", "harga": 22000, "tingkat_kepahitan": 2, "stok": 120, "rekomendasi_barista": True},
    {"nama_menu": "Americano Dingin", "kategori": "Kopi", "harga": 20000, "tingkat_kepahitan": 4, "stok": 80, "rekomendasi_barista": False},
    {"nama_menu": "Caramel Macchiato", "kategori": "Kopi", "harga": 28000, "tingkat_kepahitan": 2, "stok": 60, "rekomendasi_barista": True},
    {"nama_menu": "Tubruk Robusta Temanggung", "kategori": "Kopi", "harga": 16000, "tingkat_kepahitan": 5, "stok": 90, "rekomendasi_barista": True},
    {"nama_menu": "V60 Arabika Menoreh", "kategori": "Kopi", "harga": 26000, "tingkat_kepahitan": 3, "stok": 40, "rekomendasi_barista": False},
    
    # Kategori Makanan (memiliki 'pedas_level')
    {"nama_menu": "Cireng Bumbu Rujak", "kategori": "Makanan", "harga": 15000, "pedas_level": 2, "stok": 45, "rekomendasi_barista": True},
    {"nama_menu": "Roti Bakar Cokelat Keju", "kategori": "Makanan", "harga": 18000, "pedas_level": 0, "stok": 35, "rekomendasi_barista": False},
    {"nama_menu": "Nasi Goreng Mercon", "kategori": "Makanan", "harga": 28000, "pedas_level": 3, "stok": 25, "rekomendasi_barista": True},
    {"nama_menu": "Kentang Goreng Bolognese", "kategori": "Makanan", "harga": 22000, "pedas_level": 1, "stok": 50, "rekomendasi_barista": False},
    {"nama_menu": "Tahu Bakso Pedas Manis", "kategori": "Makanan", "harga": 17000, "pedas_level": 2, "stok": 40, "rekomendasi_barista": True},
    {"nama_menu": "Singkong Goreng Garlic", "kategori": "Makanan", "harga": 14000, "pedas_level": 0, "stok": 30, "rekomendasi_barista": False}
]

hasil_insert = koleksi_menu.insert_many(menu_kopi_nusantara)
print(f"✅ Berhasil menyisipkan {len(hasil_insert.inserted_ids)} dokumen menu ke database 'kopi_nusantara'.")
print("Total menu dalam katalog:", koleksi_menu.count_documents({}))

✅ Berhasil menyisipkan 12 dokumen menu ke database 'kopi_nusantara'.
Total menu dalam katalog: 12


## Bagian B. Kueri Bertingkat (Read)

In [4]:
# 1. Kopi tingkat kepahitan > 3
for m in koleksi_menu.find({"kategori": "Kopi", "tingkat_kepahitan": {"$gt": 3}}):
    print(f"- {m['nama_menu']} | Kepahitan: {m['tingkat_kepahitan']}/5")
print()

# 2. Harga antara 15.000 - 30.000
for m in koleksi_menu.find({"harga": {"$gte": 15000, "$lte": 30000}}):
    print(f"- {m['nama_menu']} -> Rp{m['harga']:,}")
print()

# 3. Menu tanpa atribut pedas_level
for m in koleksi_menu.find({"pedas_level": {"$exists": False}}):
    print(f"- {m['nama_menu']} (Kategori: {m['kategori']})")

- Espresso Nusantara | Kepahitan: 5/5
- Americano Dingin | Kepahitan: 4/5
- Tubruk Robusta Temanggung | Kepahitan: 5/5

- Espresso Nusantara -> Rp18,000
- Kopi Susu Gula Aren -> Rp22,000
- Americano Dingin -> Rp20,000
- Caramel Macchiato -> Rp28,000
- Tubruk Robusta Temanggung -> Rp16,000
- V60 Arabika Menoreh -> Rp26,000
- Cireng Bumbu Rujak -> Rp15,000
- Roti Bakar Cokelat Keju -> Rp18,000
- Nasi Goreng Mercon -> Rp28,000
- Kentang Goreng Bolognese -> Rp22,000
- Tahu Bakso Pedas Manis -> Rp17,000

- Espresso Nusantara (Kategori: Kopi)
- Kopi Susu Gula Aren (Kategori: Kopi)
- Americano Dingin (Kategori: Kopi)
- Caramel Macchiato (Kategori: Kopi)
- Tubruk Robusta Temanggung (Kategori: Kopi)
- V60 Arabika Menoreh (Kategori: Kopi)


## Bagian C. Pembaruan Data (Update)

In [7]:
# Set status Tersedia ke semua menu
koleksi_menu.update_many({}, {"$set": {"status": "Tersedia"}})

# Ubah 2 menu menjadi Habis
koleksi_menu.update_one({"nama_menu": "V60 Arabika Menoreh"}, {"$set": {"status": "Habis"}})
koleksi_menu.update_one({"nama_menu": "Nasi Goreng Mercon"}, {"$set": {"status": "Habis"}})

UpdateResult({'n': 1, 'nModified': 1, 'ok': 1.0, 'updatedExisting': True}, acknowledged=True)

## Bagian D. Integrasi ke Apache Spark DataFrame

In [11]:
from spark_init import spark, F
import pandas as pd

# Konversi Mongo -> Pandas -> Spark
docs = list(koleksi_menu.find({}))
df_p = pd.DataFrame(docs)
df_p["_id"] = df_p["_id"].astype(str)

df_spark_menu = spark.createDataFrame(df_p)

print("=== SKEMA HASIL INTEGRASI MONGODB KE SPARK ===")
df_spark_menu.printSchema()

print("=== RATA-RATA HARGA PER KATEGORI MENU ===")
df_spark_menu.groupBy("kategori").agg(
    F.avg("harga").alias("rata_rata_harga"),
    F.count("nama_menu").alias("total_item")
).show()

=== SKEMA HASIL INTEGRASI MONGODB KE SPARK ===
root
 |-- _id: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- kategori: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- tingkat_kepahitan: double (nullable = true)
 |-- stok: long (nullable = true)
 |-- rekomendasi_barista: boolean (nullable = true)
 |-- status: string (nullable = true)
 |-- pedas_level: double (nullable = true)

=== RATA-RATA HARGA PER KATEGORI MENU ===
+--------+------------------+----------+
|kategori|   rata_rata_harga|total_item|
+--------+------------------+----------+
|    Kopi|21666.666666666668|         6|
| Makanan|           19000.0|         6|
+--------+------------------+----------+

